In [1]:
%run thinspline_0502_1137.py

In [2]:
X = [103.715485, 103.711008, 103.88944, 103.881373]
Y = [1.445079, 1.305168, 1.320352, 1.402173]
px = [15, 13, 80, 75]
py = [100, 15, 20, 80]
gcps_lst = list(zip(X, Y, px, py))

print("✓ Module loaded")
print(f"✓ {len(gcps_lst)} GCPs defined")

✓ Module loaded
✓ 4 GCPs defined


In [7]:
# Step 2: Create test image with tracking pixel at (80, 90)
# Create a 150x150 test image
test_width = 150
test_height = 150
test_data = np.full((test_height, test_width), 100, dtype=np.float32)

gcp_vals = [200, 400, 600, 800]
for (X_, Y_, p_x, p_y), val in zip(gcps_lst, gcp_vals):
    test_data[p_y, p_x] = val

# Place tracking pixel at (80, 90) with value 1000
test_data[90, 80] = 1000

# Save as GeoTIFF with GCPs
driver = gdal.GetDriverByName('GTiff')
test_ds = driver.Create('/mnt/c/Users/nzhihao/Desktop/dso/Affine_Transformation/tif/test1.tif', test_width, test_height, 1, gdal.GDT_Float32)

# Add GCPs
gcp_list = [gdal.GCP(X_, Y_, 0, p_x, p_y) for X_, Y_, p_x, p_y in gcps_lst]
srs = osr.SpatialReference()
srs.ImportFromEPSG(4326)
test_ds.SetGCPs(gcp_list, srs.ExportToWkt())

# Write data
band = test_ds.GetRasterBand(1)
band.WriteArray(test_data)
band.SetNoDataValue(-9999)
band = None
test_ds = None

print(f"✓ Test image created: test1.tif")
print(f"  Size: {test_width}x{test_height}")
print(f"  Background value: 100")
print(f"  Tracking pixel at (80, 90): value = 1000")

✓ Test image created: test1.tif
  Size: 150x150
  Background value: 100
  Tracking pixel at (80, 90): value = 1000


In [9]:
# gdalwarp --debug ON -t_srs EPSG:4326 -co BIGTIFF=YES -r near -ot Float64 -te 103 1.30 104 1.45 -tr 0.0026 0.0026 -to SRC_METHOD=GCP_POLYNOMIAL -to MAX_GCP_ORDER=1 test1.tif gdalw1.tif

In [ ]:
result_ds = warp_to_file(
    src_filename='/mnt/c/Users/nzhihao/Desktop/dso/Affine_Transformation/tif/test1.tif',
    dst_filename='/mnt/c/Users/nzhihao/Desktop/dso/Affine_Transformation/tif/py_warp2.tif',
    gcps_list=gcps_lst,
    dst_extent=[103, 1.3, 104, 1.45],
    dst_pixel_size=[0.0026, 0.0026],
    bRefine = False,
    dfTolerance = -1,
    nGCPmin = 4,
    nodata_value = 0,
    reopen_output=False,
    debug=True,
    use_tps = False
)

Output path: /mnt/c/Users/nzhihao/Desktop/dso/Affine_Transformation/tif/py_warp3.tif
Output dir exists: True
Output dir writable: True
Output dir write test: OK
Destination grid: 385x58 pixels
  Extent: (103, 1.3, 104, 1.45)
  Pixel size: (0.0026, 0.0026)
Source image: 150x150 pixels
Using Affine transformer

Transformed source coordinates:
  X range: [-248.91, 119.94]
  Y range: [5.27, 106.53]

Sample transformations (dest -> src):
  Dest pixel center (0+0.5, 0+0.5) -> Src pixel (-248.91, 100.02)
  Dest pixel center (192+0.5, 29+0.5) -> Src pixel (-64.48, 55.07)
  Dest pixel center (384+0.5, 57+0.5) -> Src pixel (119.94, 11.78)

  Valid pixels: 7258 / 22330 (32.5%)
  Valid pixel range in dest: X=[259, 384], Y=[0, 57]
  Corresponding src coords: X=[0, 119], Y=[9, 106]

  Sample source values at valid coordinates:
    Dest(266,21) <- Src(6,69) = 100.0
    Dest(328,30) <- Src(65,55) = 100.0
    Dest(336,27) <- Src(73,60) = 100.0
    Dest(356,31) <- Src(92,54) = 100.0
    Dest(346,6) <- S

In [ ]:
# gdalwarp --debug ON -t_srs EPSG:4326 -co BIGTIFF=YES -r near -ot Float64 -te 103 1.30 104 1.45 -tr 0.0026 0.0026 test1.tif gdalw1.tif

In [ ]:
result_ds = warp_to_file(
    src_filename='/mnt/c/Users/nzhihao/Desktop/dso/Affine_Transformation/tif/test1.tif',
    dst_filename='/mnt/c/Users/nzhihao/Desktop/dso/Affine_Transformation/tif/py_warp2.tif',
    gcps_list=gcps_lst,
    dst_extent=[103, 1.3, 104, 1.45],
    dst_pixel_size=[0.0026, 0.0026],
    bRefine = False,
    dfTolerance = -1,
    nGCPmin = 4,
    nodata_value = 0,
    reopen_output=False,
    debug=True,
    use_tps = True
)